## Ejemplo MLP para regresión
<div style="border-style:groove;border-width:thin;padding:10px">
En este ejercicio vamos a resolver un problema sencillo de regresión con una red neuronal. Para ello vamos a utilizar un dataset que ya esté preparado. Solo tiene características numéricas y no hay valores nulos. Tendréis que tener en cuenta esto cuando vayáis a trabajar con datasets no preparados anteriormente.
</div>

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

2024-03-19 12:17:29.359941: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-19 12:17:30.267618: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


<div style="border-style:groove;border-width:thin;padding:10px">
En primer lugar cargamos los datos, creamos los conjuntos de train y test y escalamos los datos (necesario para usar descenso de gradiente). También vamos a dividir el conjunto de train en dos para poder usar un subconjunto como validación.
</div>

In [2]:
housing = fetch_california_housing()
scaler = StandardScaler()
X = scaler.fit_transform(housing.data)
X_train_full, X_test, y_train_full, y_test = train_test_split(
X, housing.target)
X_train, X_valid, y_train, y_valid = train_test_split(
X_train_full, y_train_full)
X.shape

(20640, 8)

<div style="border-style:groove;border-width:thin;padding:10px">
Veamos las diferencias que presentará nuestro modelo con respecto al que hicimos en clasificación.
    <ul>
        <li>La capa de salida tiene una sola neurona, ya que tenemos que calcular un solo valor.</li>
        <li>Esta capa final no tendrá ninguna función de activación.</li>
        <li>La función de pérdida será el Error Cuadrático Medio (MSE).</li>
    </ul>
    <p>Dado que el problema es bastante sencillo y los datos son ruidosos vamos a poner una sola capa oculta con menos neuronas para evitar el Overfitting.</p>        
</div>

In [3]:
#Creamos el modelo secuencial de red neuronal y pasamos las capas que se van a usar en el propio constructor.
#Como tenemos 8 entradas empezamos con una capa inicial de 6 neuronas (aprox 2/3 del número de entradas)
#Son capas de tipo denso (todas las salidas de una capa se conectan con todas las entradas de la siguiente)
#El tipo de activación se saca de la tabla de teoría.
model = keras.models.Sequential([
keras.layers.Dense(6, activation="relu", input_shape=X_train.shape[1:]),
keras.layers.Dense(1)
])

#Compilamos indicando el tipo de función de pérdida y el optimizador (descenso de gradiente, en este caso)
model.compile(loss="mean_squared_error", optimizer="sgd",metrics=['mse'])

#Entrenamos pasádnole que conjunto de validación tiene que usar. También se podría usar el parámetro validation_split=0.1, por ejemplo, para que coja el 10%.
history = model.fit(X_train, y_train, epochs=20,
validation_data=(X_valid, y_valid))
#Para ver que tal se ha hecho podemos usar el método evaluate:
mse_test = model.evaluate(X_test, y_test)
#Cogemos un dato "nuevo" para ver como ser haría una predicción.
X_new = X_test[:3] 
y_pred = model.predict(X_new)

/home/loren/anaconda3/lib/python3.11/site-packages/keras/src/layers/core/dense.py:85: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2024-03-19 12:17:32.797923: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-03-19 12:17:32.835507: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pc

Epoch 1/20


I0000 00:00:1710847053.345270   39109 service.cc:145] XLA service 0x73bf90005df0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1710847053.345294   39109 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 2070 with Max-Q Design, Compute Capability 7.5
2024-03-19 12:17:33.357552: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-03-19 12:17:33.395767: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


208/363 ━━━━━━━━━━━━━━━━━━━━ 0s 730us/step - loss: 2.1374 - mse: 2.1374

I0000 00:00:1710847053.587513   39109 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.7121 - mse: 1.7121 - val_loss: 0.7432 - val_mse: 0.7433
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6174 - mse: 0.6174 - val_loss: 0.5930 - val_mse: 0.5931
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5239 - mse: 0.5239 - val_loss: 0.5382 - val_mse: 0.5383
Epoch 4/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4892 - mse: 0.4892 - val_loss: 0.5245 - val_mse: 0.5246
Epoch 5/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4634 - mse: 0.4634 - val_loss: 0.5211 - val_mse: 0.5212
Epoch 6/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4519 - mse: 0.4519 - val_loss: 0.5114 - val_mse: 0.5115
Epoch 7/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4517 - mse: 0.4517 - val_loss: 0.5033 - val_mse: 0.5034
Epoch 8/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.4389 - mse: 0.4389 - val_loss: 0.4999 - val_mse: 0.5000
Epoch 9/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4251 

<div style="border-style:groove;border-width:thin;padding:10px">
Vamos a ver ahora algunas utilidades. En primer lugar, veremos como guardar el modelo que estamos entrenando y como cargarlo.
</div>

In [5]:
model.save("my_keras_model.keras")

In [8]:
model = keras.models.load_model("my_keras_model.keras")

<div style="border-style:groove;border-width:thin;padding:10px">
Otra funcionalidad muy útil es el early stopping. Configuramos el modelo para entrenar durante muchas épocas y creamos un callback de tipo early stopping. Lo que hará este callback es detener el progreso cuando durante un determinado número de épocas no se aprecie mejora. Además, guardará los datos del último antes de no mejorar más. 
</div>

In [9]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=1000,
validation_data=(X_valid, y_valid),
callbacks=[early_stopping_cb])

Epoch 1/1000


363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4012 - mse: 0.4012 - val_loss: 0.4635 - val_mse: 0.4635
Epoch 2/1000
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4021 - mse: 0.4021 - val_loss: 0.4713 - val_mse: 0.4713
Epoch 3/1000
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3930 - mse: 0.3930 - val_loss: 0.4707 - val_mse: 0.4708
Epoch 4/1000
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3982 - mse: 0.3982 - val_loss: 0.4679 - val_mse: 0.4680
Epoch 5/1000
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3882 - mse: 0.3882 - val_loss: 0.4643 - val_mse: 0.4644
Epoch 6/1000
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3883 - mse: 0.3883 - val_loss: 0.4611 - val_mse: 0.4612
Epoch 7/1000
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3832 - mse: 0.3832 - val_loss: 0.4579 - val_mse: 0.4579
Epoch 8/1000
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3928 - mse: 0.3928 - val_loss: 0.4552 - val_mse: 0.4553
Epoch 9/1000
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step